<span style="font-size:11px">

##### 감성 분석 -> 머신러닝
- 데이터셋 : 전처리
- BoW 모델
    - 단어를 특성 벡터로 변환
    - tf-idf 단어 적합성 평가
    - 텍스트 데이터 정제
    - 문서를 토큰으로 나누기
- LogisticRegression

In [ ]:
# 데이터셋
# http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

In [ ]:
#negative 파일 target 0 지정
import pandas as pd
from glob import glob
file_lists = glob('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\train\\neg\\*.txt')
pd_lists = []
for file_path in file_lists[:500]:
    with open(file_path, 'r', encoding='utf-8') as f:
        # print(f.read())
        data = {
            'review' : f.read(),
            'target' : 0
            }
        df = pd.DataFrame([data])
        pd_lists.append(df)
train_neg_df = pd.concat(pd_lists,ignore_index=True)
train_neg_df.head()

,review,target
0,Story of a man who has unnatural feelings for ...,0
0,Airport '77 starts as a brand new luxury 747 p...,0
0,This film lacked something I couldn't put my f...,0
0,"Sorry everyone,,, I know this is supposed to b...",0
0,When I was little my parents took me along to ...,0


In [5]:
#positive 파일 target 1 지정
import pandas as pd
from glob import glob
file_lists = glob('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\train\\pos\\*.txt')
pd_lists = []
for file_path in file_lists[:500]:
    with open(file_path, 'r', encoding='utf-8') as f:
        # print(f.read())
        data = {
            'review' : f.read(),
            'target' : 1
            }
        df = pd.DataFrame([data])
        pd_lists.append(df)
train_pos_df = pd.concat(pd_lists, ignore_index=True)
train_pos_df.head()

,review,target
0,Bromwell High is a cartoon comedy. It ran at t...,1
1,Homelessness (or Houselessness as George Carli...,1
2,Brilliant over-acting by Lesley Ann Warren. Be...,1
3,This is easily the most underrated film inn th...,1
4,This is not the typical Mel Brooks film. It wa...,1


In [ ]:
train_df = pd.concat([train_neg_df, train_pos_df], ignore_index=True)

,review,target
0,Story of a man who has unnatural feelings for ...,0
0,Airport '77 starts as a brand new luxury 747 p...,0
0,This film lacked something I couldn't put my f...,0
0,"Sorry everyone,,, I know this is supposed to b...",0
0,When I was little my parents took me along to ...,0
...,...,...
0,i watched this movie 10 years ago. and have wa...,1
0,There are many people in our lives that we mee...,1
0,"If you are a traveller, if there is a fire bur...",1
0,I had never heard of this film before a couple...,1


In [ ]:
# 상기 ignore_index=True 안했을 경우, 인덱스 새로 주기
train_df.reset_index(drop=True, inplace=True)
train_df

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0
...,...,...
995,i watched this movie 10 years ago. and have wa...,1
996,There are many people in our lives that we mee...,1
997,"If you are a traveller, if there is a fire bur...",1
998,I had never heard of this film before a couple...,1


In [24]:
train_df.to_csv('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\movie_data.csv', index=False, encoding='utf-8')

In [ ]:
df = pd.read_csv('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\movie_data.csv')
df

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0
...,...,...
995,i watched this movie 10 years ago. and have wa...,1
996,There are many people in our lives that we mee...,1
997,"If you are a traveller, if there is a fire bur...",1
998,I had never heard of this film before a couple...,1


<span style="font-size:11px">

##### BoW (Bag of Words) 모델
- 단어의 등장횟수를 카운트
- 전체 훈련데이터에서 모든 고유한 단어(토큰)로 어휘 사전
- 각 문서(리뷰데이터)를 사전을 기준으로 벡터화. 
<br> ex. N번째 단어가 문서에서 3번 나오면 벡터의 N번째값이 3이된다.
<br>     문서1 : "나는 영화가 좋다"
<br>     문서2 : "나는 영화가 싫다"
<br>     사전 : {'나는':0, '영화가':1, '좋다':2, '싫다':3}

    - 벡터화는 사전의 크기만큼 모든 문장의 길이를 동일하게
<br>     --> 문서1 벡터화 : [0,1,2] --> [1,1,1,0] (나는 | 영화가 | 좋다 | 싫다) 에 유무에 따라 표기
<br>     --> 문서2 벡터화 : [0,1,3] --> [1,1,0,1] (나는 | 영화가 | 좋다 | 싫다) 에 유무에 따라 표기
- CountVectorizer(카운트 벡터라이저) 는 BoW (Bag of Words) 모델을 파이썬에서 자동으로 만들어주는 도구

In [1]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
count = CountVectorizer()
docs = np.array ([
    "The sun is shining",
    "The weather is sweet"
])
bag = count.fit_transform(docs)

In [2]:
count.vocabulary_

{'the': 4, 'sun': 2, 'is': 0, 'shining': 1, 'weather': 5, 'sweet': 3}

<span style="font-size:11px">

#### TF-IDF (Term Frequency - Inverse Document Frequency)
- BoW를 보완하면서 좀더 정교한 텍스트 벡터화 방식
- TF-IDF
    - 특정문서에서 자주 등장하지만 전체 문장에서 드물게 등장하는 단어에 높은 가중치를 부여
    - 문서 및 문장을 대표하는 핵심단어를 찾는데 유용
    - BoW보다 단어의 중요도를 반영
    - 문서 간 유사도 계산에 강점
    - sklearn에서 TfidfVectorizer로 쉽게 구현 가능

- TF (단어빈도) 
    - 특정문서d에서 단어t가 얼마나 자주 등장하는지 나타냄
    - TF(t,d) = 문서d에서 단어t등장 횟수 / 문서d의 전체 단어수

- IDF (역문서 빈도)
    - 특정 단어가 전체 문서에서 얼마나 희귀한지 나타냄
    - 자주 등장하는 평범한 단어(ex. '나는', '그리고')는 낮은 가중치
    - IDF (t,D) = log (  총문서수 |D| / 단어t를 포함하는 총 문서의 수 df(t)  )
        - log: 단어의 희귀성을 너무 과하게 반영하지 않도록 스케일링
    - sklearn 의 TfidVectorizer
        - log (  1 + 총문서수 |D| / 1 + 단어t를 포함하는 총 문서의 수 df(t) ) + 1

- TF-IDF(t,d,D) = TF(t,d) X IDF (t,D)
    - "나는"
    - TF : 리뷰에 3번 나옴 (가중치 높음)
    - IDF : 전체 10,000개 리뷰 중에 9000개 나옴 (가중치 매우 낮음)
    - TF-IDF : 가중치 높음 X 가중치 매우 낮음 = 낮음 (중요도가 낮음)
    - "명작"
    - TF : 리뷰에 2번 나옴 (가중치 높음)
    - IDF : 전체 10,000개 리뷰 중에 50개 나옴 (가중치 매우 높음)
    - TF-IDF : 가중치 높음 X 가중치 매우 높음 = 높음 (중요도가 높음/ 핵심단어)

In [ ]:
import pandas as pd
df = pd.read_csv('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\movie_data.csv')
df

# 데이터 정제 : html tag와 같은 불필요한 string이 보임. 특수기호 (< - ' ) 기타 등등
df.review[0] 
# 정제 대상 찾기 : 영문 공백 . , 

import re
def preprocessor(s):
    # 1. 영문, 공백, ., , 만 남기기
    clean = re.sub(r'[^A-Za-z\s.,]+', '', s)
    # 2. 연속된 마침표(...)를 마침표 하나로
    clean = re.sub(r'\.{2,}', '.', clean)
    # 3. 연속된 공백 처리
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

In [5]:
df['review']=df.review.apply(preprocessor)

In [ ]:
%conda install -c conda-forge nltk    

In [16]:
# 문서를 토큰으로 나누기
from nltk.stem.porter import PorterStemmer   # 어간을 추출하는 작업을 함.

def tokenizer (text):   # tokenizer 공백을 기준으로 단순 text split
    return text.split()

porter = PorterStemmer()
def tokenizer_porter (text):
    return [porter.stem(word) for word in text.split()]

# 어간 추출 : stemming 단어의 접미사 -s -es -ing -ed 등등 을 강제로 제거해서 단어의 원형을 찾는 과정

df.review[0]
print (tokenizer(df.review[0][:100]))
print (tokenizer_porter(df.review[0][:100]))



['Story', 'of', 'a', 'man', 'who', 'has', 'unnatural', 'feelings', 'for', 'a', 'pig.', 'Starts', 'out', 'with', 'a', 'opening', 'scene', 'that', 'is', 'a', 'terri']
['stori', 'of', 'a', 'man', 'who', 'ha', 'unnatur', 'feel', 'for', 'a', 'pig.', 'start', 'out', 'with', 'a', 'open', 'scene', 'that', 'is', 'a', 'terri']


In [34]:
# 불용어 제거를 위해 불용어 확인
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')  #불용어 사전 다운로드
stops = stopwords.words('english')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [19]:
df.head() # 전처리 완료 정규식을 이용한

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport starts as a brand new luxury plane is ...,0
2,This film lacked something I couldnt put my fi...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0


In [ ]:
from sklearn.model_selection import train_test_split
X = df.review  #2차원 만들면 오류남. 1차원 해야함
y = df.target
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, stratify=y, random_state=42)



In [39]:
# 어간 추출 (PorterStemmer) --> stopwords 불용어 단어 제거
# 불용어 제거를 위해 불용어 확인

from nltk.corpus import stopwords
stops = stopwords.words('english')

porter = PorterStemmer()
def tokenizer_porter (text):
    return [porter.stem(word) for word in text.split() if word not in stops]


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline



In [40]:
tfidf = TfidfVectorizer (
    tokenizer=tokenizer_porter,
    ngram_range=(1,1)  # (1,1) 유니그램(unigramm, 단일단어)만 사용
)

pipe = Pipeline([
    ('tfidf', tfidf),
    ('clf', LogisticRegression())
])

pipe.fit(X_train, y_train)


c:\Users\playdata2\miniconda3\envs\deep\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,<function tok...0029399086340>


In [41]:
X_train.shape, y_train.shape

((800,), (800,))

In [43]:
print(X_test.to_numpy()[0])
print(y_test.to_numpy()[0])

When I first saw A Cry in the Dark, I had no idea what the plot was. But when I saw it, I was shocked at what it portrayed. When I saw it a second time in an Australian Cinema class, I realized a second point communication issues. You see, when a dingo snatched Lindy Chamberlains Meryl Streep baby, she and her husband Michael Sam Neill were griefstricken but didnt show it. As Seventh Day Adventists, they believed that God willed this to happen, and so they couldnt mourn it. But when people all over Australia saw their lack of sadness, everyone started believing that Lindy did it herself.br br The point is, the wrong message got communicated to the public, and it turned people against Lindy. Even though this was a pure accident, it still happened. It may be one of the biggest disasters resulting from the existence of mass media, regardless of any media outlets political views.br br As for the performances, Streep does a very good job with an Australian accent no surprise there, and Sam 

In [45]:
pipe.predict(X_test)

array([1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0,
       0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0,
       1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0,
       1, 1])

In [46]:
y_test.values

array([1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0,
       0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0,
       0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0,
       1, 1])

In [48]:
print (f' train_score : {pipe.score(X_train,y_train)}, test_score: {pipe.score(X_test,y_test)}')
from sklearn.metrics import classification_report
print (classification_report(y_test, pipe.predict(X_test)))


 train_score : 0.99625, test_score: 0.935
              precision    recall  f1-score   support

           0       0.92      0.95      0.94       100
           1       0.95      0.92      0.93       100

    accuracy                           0.94       200
   macro avg       0.94      0.94      0.93       200
weighted avg       0.94      0.94      0.93       200



In [ ]:
### 총 실습

# 원본 데이터 로드
    # train 폴더에 있는 데이터로 학습 - 적당한 크기로
    # 정답은 test 폴더에 있는 문장으로 평가
# 토크나이저 함수를 정의
    # 텍스트 전처리
    # 공백을 기준으로 단어단위로 분리
    # 영어는 전부 소문자로 변환
    # 어간 추출
    # 불용어 제거
# TFIDF 를 정의
    # 토크나이저 매개변수 = 토크나이저 함수
    # ngram (1,1)
# 파이프라인으로 tfidf, 머신러닝
# 파이프라인으로 학습
# 파이프라인으로 평가 (classification_report)
# 과적합 여부 확인
    # train데이터와 test 데이터로 성능을 비교 (score)


In [ ]:
#train _ negative 파일 target 0 지정
import pandas as pd
from glob import glob
file_lists = glob('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\train\\neg\\*.txt')
pd_lists = []
for file_path in file_lists:
    with open(file_path, 'r', encoding='utf-8') as f:
        # print(f.read())
        data = {
            'review' : f.read(),
            'target' : 0
            }
        df = pd.DataFrame([data])
        pd_lists.append(df)
train_neg1_df = pd.concat(pd_lists,ignore_index=True)
train_neg1_df.head()

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0


In [ ]:
#train _ positive 파일 target 1 지정
import pandas as pd
from glob import glob
file_lists = glob('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\train\\pos\\*.txt')
pd_lists = []
for file_path in file_lists:
    with open(file_path, 'r', encoding='utf-8') as f:
        # print(f.read())
        data = {
            'review' : f.read(),
            'target' : 1
            }
        df = pd.DataFrame([data])
        pd_lists.append(df)
train_pos1_df = pd.concat(pd_lists,ignore_index=True)
train_pos1_df.head()

,review,target
0,Bromwell High is a cartoon comedy. It ran at t...,1
1,Homelessness (or Houselessness as George Carli...,1
2,Brilliant over-acting by Lesley Ann Warren. Be...,1
3,This is easily the most underrated film inn th...,1
4,This is not the typical Mel Brooks film. It wa...,1


In [ ]:
# train negative 와 positive 리뷰 합치기
train_df2 = pd.concat([train_neg1_df, train_pos1_df], ignore_index=True)
train_df2

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0
...,...,...
24995,"Seeing as the vote average was pretty low, and...",1
24996,"The plot had some wretched, unbelievable twist...",1
24997,I am amazed at how this movie(and most others ...,1
24998,A Christmas Together actually came before my t...,1


In [53]:
# 합친 train_df2 파일 csv로 저장하기
train_df2.to_csv('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\movie_data_all.csv', index=False, encoding='utf-8')

In [54]:
# 저장된 movie_data_all.csv 파일 불러오기
df2 = pd.read_csv('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\movie_data_all.csv')
df2

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0
...,...,...
24995,"Seeing as the vote average was pretty low, and...",1
24996,"The plot had some wretched, unbelievable twist...",1
24997,I am amazed at how this movie(and most others ...,1
24998,A Christmas Together actually came before my t...,1


In [55]:
#test 파일 : negative 파일 target 0 지정
import pandas as pd
from glob import glob
file_lists = glob('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\test\\neg\\*.txt')
pd_lists = []
for file_path in file_lists:
    with open(file_path, 'r', encoding='utf-8') as f:
        # print(f.read())
        data = {
            'review' : f.read(),
            'target' : 0
            }
        df = pd.DataFrame([data])
        pd_lists.append(df)
test_neg1_df = pd.concat(pd_lists,ignore_index=True)
test_neg1_df.head()

,review,target
0,Once again Mr. Costner has dragged out a movie...,0
1,This is an example of why the majority of acti...,0
2,"First of all I hate those moronic rappers, who...",0
3,Not even the Beatles could write songs everyon...,0
4,Brass pictures (movies is not a fitting word f...,0


In [56]:
#test _ positive 파일 target 1 지정
import pandas as pd
from glob import glob
file_lists = glob('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\test\\pos\\*.txt')
pd_lists = []
for file_path in file_lists:
    with open(file_path, 'r', encoding='utf-8') as f:
        # print(f.read())
        data = {
            'review' : f.read(),
            'target' : 1
            }
        df = pd.DataFrame([data])
        pd_lists.append(df)
test_pos1_df = pd.concat(pd_lists,ignore_index=True)
test_pos1_df.head()

,review,target
0,I went and saw this movie last night after bei...,1
1,Actor turned director Bill Paxton follows up h...,1
2,As a recreational golfer with some knowledge o...,1
3,"I saw this film in a sneak preview, and it is ...",1
4,Bill Paxton has taken the true story of the 19...,1


In [57]:
# test negative 와 positive 리뷰 합치기
test_df2 = pd.concat([test_neg1_df, test_pos1_df], ignore_index=True)
test_df2

,review,target
0,Once again Mr. Costner has dragged out a movie...,0
1,This is an example of why the majority of acti...,0
2,"First of all I hate those moronic rappers, who...",0
3,Not even the Beatles could write songs everyon...,0
4,Brass pictures (movies is not a fitting word f...,0
...,...,...
24995,I was extraordinarily impressed by this film. ...,1
24996,"Although I'm not a golf fan, I attended a snea...",1
24997,"From the start of ""The Edge Of Love"", the view...",1
24998,"This movie, with all its complexity and subtle...",1


In [58]:
# 합친 test_df2 파일 csv로 저장하기
test_df2.to_csv('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\movie_data_test_all.csv', index=False, encoding='utf-8')

In [17]:
# 저장된 movie_data_all.csv 파일 불러오기
import pandas as pd
# train 데이터
df2 = pd.read_csv('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\movie_data_all.csv')
df2 

# test 데이터
test_df = pd.read_csv('D:\\0python_SNC\\17machin-learning\\data\\movie_aclImdb\\movie_data_test_all.csv')
test_df

# 데이터 전처리 함수 만들기: 영문 공백 . , 

import re
from nltk.stem.porter import PorterStemmer
porter = PorterStemmer()
from nltk.corpus import stopwords
stops = stopwords.words('english')

# from sklearn.model_selection import train_test_split
# train, test 각각 파일 지정하기 때문에 필요없음.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def custom_tokenizer(text):
    # 전처리
    # 1. 영문, 공백, ., , 만 남기기
    clean = re.sub(r'[^A-Za-z\s.,]+', '', text)
    # 2. 연속된 마침표(...)를 마침표 하나로
    clean = re.sub(r'\.{2,}', '.', clean)
    # 3. 연속된 공백 처리
    clean = re.sub(r'\s+', ' ', clean).strip()
    
        # 단어분리-어간분리-불용어 제거
    return [porter.stem(word) for word in clean.split() if word not in stops]


X_train = df2.review  
y_train = df2.target

X_test = test_df.review
y_test = test_df.target



tfidf = TfidfVectorizer(
    tokenizer=custom_tokenizer,
    ngram_range=(1,1),
    token_pattern=None
)

pipe = Pipeline([
    ('tfid',tfidf),
    ('clf',LogisticRegression())
])


#학습
pipe.fit(X_train, y_train)

from sklearn.metrics import classification_report
print(f'train report {classification_report(y_train, pipe.predict(X_train))}')
print(f'train report {classification_report(y_test, pipe.predict(X_test))}')


train report               precision    recall  f1-score   support

           0       0.94      0.92      0.93     12500
           1       0.93      0.94      0.93     12500

    accuracy                           0.93     25000
   macro avg       0.93      0.93      0.93     25000
weighted avg       0.93      0.93      0.93     25000

train report               precision    recall  f1-score   support

           0       0.87      0.87      0.87     12500
           1       0.87      0.87      0.87     12500

    accuracy                           0.87     25000
   macro avg       0.87      0.87      0.87     25000
weighted avg       0.87      0.87      0.87     25000



In [18]:
###############################################################

In [ ]:
# 상기 파일 처리 관련 코드

In [ ]:
from glob import glob
import numpy as np
from tqdm import tqdm

def get_data(pattern, neg=True,to = None):
    documents = []
    if to is not None:
        for path in tqdm(glob(pattern)[ : to]):
            with open(path, 'r', encoding='utf-8') as f:
                documents.append(  (np.array(f.read()),  0 if neg else 1)   )
    else:
        for path in tqdm(glob(pattern)):
            with open(path, 'r', encoding='utf-8') as f:
                documents.append(  (np.array(f.read()),  0 if neg else 1)   )
    return documents

# "../data/movie/train/neg/*.txt"     

'감'

In [ ]:
########### 학성님 공유코드

In [ ]:
import pandas as pd
from glob import glob

test_file_list = glob('./data/movie/test/neg/*.txt')
display(len(test_file_list))
test_file_list += glob('./data/movie/test/pos/*.txt')
# display(test_file_list[:5], len(test_file_list))

pd_list = []

for path in test_file_list :
    with open(path, 'r', encoding='utf-8') as f :
        data = {
            'review' : f.read()
            , 'target' : 0 if 'neg' in path else 1
        }
        df = pd.DataFrame([data])
        pd_list.append(df)

test_df = pd.concat(pd_list, ignore_index=True)
display(test_df.head(), test_df.info(), test_df.describe())

train_file_list = glob('./data/movie/train/neg/*.txt')
display(len(train_file_list))
train_file_list += glob('./data/movie/train/pos/*.txt')
# display(train_file_list[:5], len(train_file_list))

pd_list = []

for path in train_file_list :
    with open(path, 'r', encoding='utf-8') as f :
        data = {
            'review' : f.read()
            , 'target' : 0 if 'neg' in path else 1
        }
        df = pd.DataFrame([data])
        pd_list.append(df)

train_df = pd.concat(pd_list, ignore_index=True)
display(train_df.head(), train_df.info(), train_df.describe())